# 04 · Ajuste de hiperparámetros (tuning)
**Modelos a ajustar (3 mejores del `03`/`03b` por PR-AUC en CV):** Embeddings + LogReg (0.946), LogisticRegression con TF-IDF (0.943) y LightGBM (0.941). Con diferencias menores que la desviación entre pliegues, se eligen los tres primeros; RandomForest (0.928) queda fuera.

**Método:** búsqueda aleatoria (el mismo muestreo de `RandomizedSearchCV`), 30 configuraciones por modelo, evaluadas con **la misma CV temporal** (`TimeSeriesSplit(5)`) y métrica que en el `03`. Se prefiere la búsqueda aleatoria a la exhaustiva porque con 4–7 hiperparámetros, algunos continuos, cubre mejor el espacio con el mismo presupuesto, y a Optuna porque 30 configuraciones no justifican la complejidad extra. Cada configuración es una corrida hija dentro de una corrida padre por modelo.

Ejecutar con *Restart* y luego *Run All* (la primera vez calcula los embeddings de nuevo y los guarda en disco; unos minutos).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, mlflow
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterSampler
from lightgbm import LGBMClassifier
from src.datos import cargar_obt
from src.pipeline import *
from src.experimento import *
sns.set_theme(style="whitegrid")
df = agregar_features(cargar_obt())
train, test = split_temporal(df, 0.2)
X, y = train[FEATURES], train[TARGET]
FECHA_CORTE = df[FECHA].max().isoformat()
configurar()
N_CONFIG = 30

## 1. Espacios de búsqueda
| Modelo | Hiperparámetro | Rango | Escala / justificación |
|---|---|---|---|
| LogReg (TF-IDF) | `C` | 0.01 – 100 | **log**: la regularización actúa de forma multiplicativa |
| | `class_weight` | None / balanced | desbalance leve (44%) |
| | `max_features` TF-IDF | 1000, 3000, 6000 | tamaño del vocabulario |
| | `ngram_range` | (1,1), (1,2) | ¿ayudan los bigramas? |
| Embeddings + LogReg | `C` | 0.001 – 100 | **log** |
| | `class_weight` | None / balanced | |
| LightGBM | `n_estimators` | 100 – 600 | lineal |
| | `learning_rate` | 0.01 – 0.3 | **log** (se compensa con n_estimators) |
| | `num_leaves` | 15 – 126 | lineal; controla la complejidad |
| | `min_child_samples` | 10 – 59 | lineal; frena el sobreajuste |
| | `subsample` / `colsample_bytree` | 0.6–1.0 / 0.4–1.0 | lineal |
| | `reg_lambda` | 0.001 – 10 | **log** |

In [ ]:
ESPACIOS = {
  "LogisticRegression": dict(familia="lineal_regularizado", base=lambda: construir_pipeline(LogisticRegression(max_iter=2000)),
      esp={"modelo__C": loguniform(1e-2, 1e2), "modelo__class_weight": [None, "balanced"],
           "prep__texto__max_features": [1000, 3000, 6000], "prep__texto__ngram_range": [(1,1), (1,2)]}),
  "Embeddings+LogReg": dict(familia="estado_del_arte", base=lambda: construir_pipeline(LogisticRegression(max_iter=2000), texto="embeddings"),
      esp={"modelo__C": loguniform(1e-3, 1e2), "modelo__class_weight": [None, "balanced"]}),
  "LightGBM": dict(familia="boosting", base=lambda: construir_pipeline(LGBMClassifier(random_state=SEED, verbose=-1, subsample_freq=1)),
      esp={"modelo__n_estimators": randint(100, 601), "modelo__learning_rate": loguniform(1e-2, 3e-1),
           "modelo__num_leaves": randint(15, 127), "modelo__min_child_samples": randint(10, 60),
           "modelo__subsample": uniform(0.6, 0.4), "modelo__colsample_bytree": uniform(0.4, 0.6),
           "modelo__reg_lambda": loguniform(1e-3, 1e1)}),
}
def limpio(d): return {k.split("__")[-1]: (float(v) if isinstance(v, np.floating) else int(v) if isinstance(v, np.integer) else str(v)) for k, v in d.items()}

## 2. Ejecución: una corrida padre por modelo y 30 corridas hijas

In [ ]:
limpiar_fase("tuning"); limpiar_fase("tuning_padre")
exp = mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO)
cand = mlflow.search_runs([exp.experiment_id]); cand = cand[cand["tags.fase"] == "candidato"].set_index("tags.mlflow.runName")
resumen, hijas = [], []
for nombre, cfg in ESPACIOS.items():
    d = cand.loc[f"candidato-{nombre}"]
    with mlflow.start_run(run_name=f"tuning-{nombre}") as padre:
        mlflow.set_tags({"fase": "tuning_padre", "familia": cfg["familia"], "commit_git": commit_git()})
        mlflow.log_params({"n_configuraciones": N_CONFIG, "busqueda": "aleatoria", "semilla": SEED})
        resultados = []
        for i, pr in enumerate(ParameterSampler(cfg["esp"], N_CONFIG, random_state=SEED)):
            pipe = clone(cfg["base"]()).set_params(**pr)
            r = correr(f"{nombre}-cfg{i:02d}", pipe, "tuning", cfg["familia"], X, y, limpio(pr), FECHA_CORTE,
                       nested=True, log_modelo=False, tags={"modelo_padre": nombre})
            r.update(limpio(pr)); r["modelo"] = nombre; resultados.append(r)
        res = pd.DataFrame(resultados); hijas.append(res)
        mejor = res.loc[res["cv_pr_auc_mean"].idxmax()]
        mejora = mejor["cv_pr_auc_mean"] - d["metrics.cv_pr_auc_mean"]
        mlflow.log_metrics({"mejor_cv_pr_auc": mejor["cv_pr_auc_mean"], "default_cv_pr_auc": d["metrics.cv_pr_auc_mean"],
                            "default_cv_std": d["metrics.cv_pr_auc_std"], "mejora_vs_default": mejora,
                            "mejor_cv_std": mejor["cv_pr_auc_std"], "mejora_supera_std": float(mejora > d["metrics.cv_pr_auc_std"])})
        hp = [k.split("__")[-1] for k in cfg["esp"]]
        mlflow.log_params({"mejor_" + c: str(mejor[c]) for c in hp})
        resumen.append({"modelo": nombre, "run_id_padre": padre.info.run_id, "default_cv": d["metrics.cv_pr_auc_mean"],
                        "default_std": d["metrics.cv_pr_auc_std"], "mejor_cv": mejor["cv_pr_auc_mean"], "mejor_std": mejor["cv_pr_auc_std"],
                        "mejora": mejora, "supera_std_pliegues": bool(mejora > d["metrics.cv_pr_auc_std"]), "mejor_cfg": mejor["nombre"]})
todas = pd.concat(hijas, ignore_index=True)
print(len(todas), "corridas hijas")

## 3. Resultados del tuning

In [ ]:
res = pd.DataFrame(resumen).set_index("modelo")
display(res[["default_cv","default_std","mejor_cv","mejor_std","mejora","supera_std_pliegues","mejor_cfg"]].round(4))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for a, (nombre, g) in zip(ax, todas.groupby("modelo")):
    a.scatter(range(len(g)), g["cv_pr_auc_mean"], color="#6b7d3a")
    a.axhline(res.loc[nombre, "default_cv"], color="#c9a227", ls="--", label="por defecto")
    a.set_title(nombre); a.set_xlabel("configuración"); a.legend()
ax[0].set_ylabel("PR-AUC CV")
plt.tight_layout(); plt.savefig("/work/docs/figs/tuning.png", dpi=150); plt.show()

In [ ]:
# Efecto de cada hiperparámetro (Spearman con PR-AUC en CV)
from scipy import stats
for nombre, g in todas.groupby("modelo"):
    num = [c for c in g.columns if c in [k.split("__")[-1] for k in ESPACIOS[nombre]["esp"]] and pd.api.types.is_numeric_dtype(g[c])]
    print(nombre, {c: round(stats.spearmanr(g[c], g["cv_pr_auc_mean"])[0], 2) for c in num})

## 4. Lectura (completar con las cifras)
- Si `mejora` ≤ desviación entre pliegues (`default_std`), **no se considera mejora**, aunque la media suba.
- Los dos mejores para la fase de selección (`05`) se eligen por PR-AUC en CV y se desempatan por costo, tiempo y explicabilidad.